## Task 3: Integration with Chatbot System
Consider how you could integrate the video captioning system with a chatbot to allow users to query video content through text prompts.
Provide a brief description of your integration strategy and any potential improvements you foresee.

### Problem observations

1. When I first started this project, I quickly realized you can't just shove an entire raw video into a language model and expect it to answer questions. I needed to break the video down into smaller pieces, otherwise the system would immediately run out of memory.

2. Additionally, even if I could get a model to summarize the whole video, a chatbot should be able to tell the user when those specific events happened.

3. I needed a way to translate visual timelines into searchable text so the chatbot wouldn't have to re-watch the video every time a user asked a question.

### Result observations

1. I am honestly really happy with how the final pipeline turned out. The semantic search is almost magical—if I ask about a "mother," the vector database successfully pulls the chunk where smolvlm saw "a woman holding a baby," even without exact keyword matches.

2. I did notice that the small VLM occasionally hallucinates minor background details (like making up a second person in the background). However, because I am using prompt injection to strictly ground gemma-4-e2b-it to only use the retrieved text, the final chatbot response feels smart, since it cites the timestamps for answering the query.

### Solution strategy & challenges

1. To fix this, my strategy was to build a Retrieval-Augmented Generation (RAG) pipeline. To save hitting the google colab GPU limits, I used a 4-bit quantized version of smolvlm. I sliced the video into time-windowed frames, had smolvlm caption each chunk, and saved those text descriptions alongside their timestamps.

2. For the memory and search layer, I embedded those captions into ChromaDB using google/embeddinggemma-300m. When a user asks a question, I retrieve the top 3 most relevant chunks. Finally, I went with standard prompt injection (instead of a complex agentic loop) to feed those top 3 results into gemma-4-e2b-it to generate the final answer.

3. The biggest challenge was definitely memory management and dependency conflicts in my environment. Also, I struggled a bit with the embedding model at first until I realized embeddinggemma requires highly specific prompt prefixes (like task: search result | query: ) to actually perform accurate searches!

4. Installing ChromaDB in Colab was a massive headache initially. The environment kept crashing due to conflicting OpenTelemetry library versions, requiring me to manually pin specific versions just to get the database to install.

5. I learned the hard way that Google Colab clears its temporary storage whenever the runtime restarts, causing me to lose my initial batches of captions. I solved this by mounting my Google Drive and saving all my JSON and ChromaDB data directly to the mounted google drive.

6. When I first ran smolvlm without sampling, the generated captions were repetitive and of completely arbitrary lengths. I had to explicitly enable sampling and dial in a low temperature to force the model to produce consistent descriptions.

7. My initial tests with standard models like MiniLM felt too slow for this pipeline. Swapping to google/embeddinggemma-300m solved this; its smaller footprint and faster execution made the database ingestion much faster without sacrificing any search accuracy.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q transformers accelerate torch torchvision yt-dlp bitsandbytes peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00


In [3]:
import os
dir = "/content/drive/MyDrive/"
os.path.exists(dir)

True

In [7]:
yt_video_link = "https://www.youtube.com/shorts/CkaWa0BJbec"

In [8]:
!yt-dlp -f "best[height<=720][ext=mp4]" \
        --recode-video mp4 \
        --postprocessor-args "ffmpeg:-c:v libx264 -preset fast -crf 23" \
        {yt_video_link} \
        -o "/content/drive/MyDrive/fixed_output.mp4"

[youtube] Extracting URL: https://www.youtube.com/shorts/CkaWa0BJbec
[youtube] CkaWa0BJbec: Downloading webpage
[youtube] CkaWa0BJbec: Downloading android vr player API JSON
[info] CkaWa0BJbec: Downloading 1 format(s): 18
[download] Destination: /content/drive/MyDrive/fixed_output.mp4
[download] 100% of    3.78MiB in 00:00:00 at 17.69MiB/s
[VideoConvertor] Not converting media file "/content/drive/MyDrive/fixed_output.mp4"; already is in target format mp4


In [ ]:
import cv2
import json
import torch
import gc
import numpy as np
from PIL import Image

VIDEO_PATH = "/content/drive/MyDrive/fixed_output.mp4"

def extract_frames_for_chunk(video_path, start_time, end_time, max_frames=4, resize=(360, 640)):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    video_duration = total_frames / fps

    # Clamp the end time if it exceeds video duration
    end_time = min(end_time, video_duration)

    # Calculate exact frame indices for this chunk
    start_frame = int(start_time * fps)
    end_frame = int(end_time * fps)
    chunk_frame_count = end_frame - start_frame

    if start_frame >= total_frames or chunk_frame_count <= 0:
        cap.release()
        return []

    # Calculate target indices for the specific chunk
    n_samples = min(max_frames, chunk_frame_count)
    target_indices = set(np.linspace(start_frame, end_frame - 1, n_samples, dtype=int).tolist())

    frames = []

    # Fast Forward
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    current_frame_idx = start_frame

    while current_frame_idx < end_frame:
        ret, frame = cap.read()
        if not ret:
            break

        if current_frame_idx in target_indices:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(rgb)
            if resize is not None:
                img = img.resize(resize)

            frames.append(img)

            if len(frames) == n_samples:
                break # exit the read loop

        current_frame_idx += 1

    cap.release()
    return frames


chunk_duration = 5 # seconds
video_database = []

# Get total duration
temp_cap = cv2.VideoCapture(VIDEO_PATH)
fps = temp_cap.get(cv2.CAP_PROP_FPS)
total_frames = int(temp_cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_duration = total_frames / fps
temp_cap.release()

print(f"Video duration {video_duration}, FPS: {fps}, total_frames: {total_frames}")

Video duration 53.1, FPS: 30.0, total_frames: 1593


In [3]:
from google.colab import userdata
userdata.get('HF_TOKEN') is not None

True

In [4]:
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [16]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "HuggingFaceTB/SmolVLM-Instruct"

nf4_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=nf4_config,
    device_map="auto",
    low_cpu_mem_usage=True
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.49G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/92.0 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [17]:
model.device

device(type='cuda', index=0)

In [23]:
import math

frames_to_extract = math.ceil(int(video_duration) / chunk_duration)
frames_to_extract

11

In [24]:
from tqdm import tqdm

for start_time in tqdm(range(0, int(video_duration), chunk_duration), desc='Extracting frames'):
    end_time = min(start_time + chunk_duration, video_duration)

    # Extract 4 evenly spaced frames from this 5-second window
    pil_frames = extract_frames_for_chunk(
        VIDEO_PATH,
        start_time,
        end_time,
        max_frames=4,
        resize=(360, 640)
    )

    if not pil_frames:
        continue

    messages = [{
        "role": "user",
        "content": [
            {"type": "image"} for _ in pil_frames
        ] + [{
            "type": "text",
            "text": "Describe the specific actions and key subjects occurring in this brief sequence of video frames."
        }]
    }]

    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=pil_frames, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.3, # low randomness
            top_k=50 # sampling of higher probability tokens
        )
        caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        clean_caption = caption.split("Assistant:")[-1].strip()

    # Structure and save the data
    chunk_data = {
        "start_time": start_time,
        "end_time": end_time,
        "caption": clean_caption
    }
    video_database.append(chunk_data)
    tqdm.write(f"Processed chunk {start_time}s to {end_time}s: {clean_caption[:50]}...")

    del pil_frames
    del inputs
    del generated_ids
    gc.collect()
    torch.cuda.empty_cache()

Extracting frames:   0%|          | 0/11 [00:37<?, ?it/s]

Processed chunk 0s to 5s: The video shows a baby girl eating an ice cream ba...


Extracting frames:   9%|▉         | 1/11 [01:18<06:15, 37.54s/it]

Processed chunk 5s to 10s: The image is a screenshot from a video showing a b...


Extracting frames:  18%|█▊        | 2/11 [01:53<05:57, 39.74s/it]

...

Extracting frames:  82%|████████▏ | 9/11 [06:12<01:14, 37.47s/it]

Processed chunk 45s to 50s: The video is a short clip showing a baby eating ic...


Extracting frames:  91%|█████████ | 10/11 [06:48<00:37, 37.68s/it]

Processed chunk 50s to 53.1s: A baby is being held by an adult and eating a spoo...


Extracting frames: 100%|██████████| 11/11 [06:48<00:00, 37.15s/it]


In [25]:
with open('/content/drive/MyDrive/timestamped_captions.json', 'w') as f:
    json.dump(video_database, f, indent=4)

print("Done! Ready for Vector DB.")

Done! Ready for Vector DB.


In [3]:
!pip install -q chromadb "opentelemetry-api==1.38.0" "opentelemetry-sdk==1.38.0" "opentelemetry-exporter-otlp-proto-common==1.38.0" "opentelemetry-exporter-otlp-proto-http==1.38.0" "opentelemetry-proto==1.38.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.2 MB/s eta 0:00:00


In [4]:
import json
import chromadb
from chromadb.utils import embedding_functions
from tqdm import tqdm

with open('/content/drive/MyDrive/timestamped_captions.json', 'r') as f:
    video_database = json.load(f)

chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/video_vector_db")

embedding_model = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="google/embeddinggemma-300m")

collection = chroma_client.get_or_create_collection(
    name="my_video_captions",
    embedding_function=embedding_model
)

documents = []
metadatas = []
ids = []

for i, chunk in tqdm(enumerate(video_database), desc='Creating vector db...'):
    documents.append(chunk["caption"])
    metadatas.append({
        "start_time": chunk["start_time"],
        "end_time": chunk["end_time"]
    })
    ids.append(f"chunk_{i}")

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Successfully added {len(documents)} video chunks to the Vector DB!")

KeyboardInterrupt: 

In [4]:
import chromadb
from chromadb.utils import embedding_functions
from IPython.display import Markdown

chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/video_vector_db")

embedding_model = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="google/embeddinggemma-300m")

collection = chroma_client.get_or_create_collection(
    name="my_video_captions",
    embedding_function=embedding_model
)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

In [5]:
from tqdm import tqdm

def search_video_database(query: str, top_k: int=3) -> str:
    """
    Searches the video caption database to find timestamps of specific events, objects, or actions.
    Use this tool whenever the user asks what happened in the video.

    Args:
        query (str): The specific action or object to search for (e.g., 'baby crying', 'chocolate ice cream').
        top_k (int): Results to fetch from chroma db (default=3)
    """
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )

    outputs = []
    for i in tqdm(range(len(results['documents'][0])), desc='Searching..'):
        clean_text = results['documents'][0][i]
        start = results['metadatas'][0][i]['start_time']
        end = results['metadatas'][0][i]['end_time']
        distance = results['distances'][0][i]

        outputs.append(f"- **Match {i+1} ({start}s to {end}s)** with distance **{distance:.4f}**:")
        outputs.append(f"- {clean_text}")
        outputs.append("----")

    return "\n".join(outputs) if outputs else "No matching events found in the video."

Markdown(search_video_database("Is a baby crying?", 4))

Searching..: 100%|██████████| 4/4 [00:00<00:00, 23269.37it/s]


- **Match 1 (20s to 25s)** with distance **0.4740**:
- A woman is holding a baby in her arms. The woman is smiling and has her hand on the baby's hand. The baby is crying. The woman is wearing a gray shirt. The background is green.
----
- **Match 2 (25s to 30s)** with distance **0.5531**:
- A woman holds a baby. The woman is smiling at the baby. The baby is looking at the camera. The baby has a pacifier in its mouth.
----
- **Match 3 (45s to 50s)** with distance **0.6154**:
- The video is a short clip showing a baby eating ice cream. The baby is being held by an adult and is positioned in a way that suggests it is eating the ice cream. The baby is making a face of surprise and seems to be enjoying the ice cream. The adult is smiling and appears to be happy about the baby eating the ice cream. The baby is wearing a diaper and is surrounded by a blanket. The background is blurred and is not clear.
----
- **Match 4 (0s to 5s)** with distance **0.6179**:
- The video shows a baby girl eating an ice cream bar. The baby is sitting in a baby carrier and is wearing a headband with a large bow on it. The baby's eyes are open and she has a neutral expression. The baby is holding the ice cream bar with both hands and is taking a bite. The ice cream bar is white and has a brown wrapper. The background is blurred.
----

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Free GPU Memory
torch.cuda.empty_cache()

model_id = "google/gemma-4-E2B-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.float16,
).eval()
llm_model

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (o_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=Fals

In [7]:
llm_model.device

device(type='cuda', index=0)

In [ ]:
from IPython.display import Markdown, display_markdown

def ask_video_chatbot(user_question: str) -> str:
    # Format the query for EmbeddingGemma
    gemma_formatted_query = f"task: search result | query: {user_question}"

    # Retrieve the top 3 most relevant chunks from ChromaDB
    context_block = search_video_database(gemma_formatted_query, 3)

    # Handle the edge case where nothing was found
    if not context_block:
        return "I couldn't find any relevant scenes in the video to answer that."

    system_prompt = (
        "You are a helpful video assistant. Answer the user's question using ONLY "
        "the provided video context. Always cite the exact timestamps in your answer. "
        "If the context does not contain the answer, say 'I cannot see that in the video.'"
    )

    user_prompt = f"Video Context:\n{context_block}\n\nUser Question: {user_question}\n"

    # Structure it as a chat conversation
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # Apply the chat template so the model understands the formatting
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt"
    )
    inputs = tokenizer(text=text, return_tensors="pt").to(llm_model.device)

    # Generate the final answer
    with torch.inference_mode():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.6,
            top_p=0.9,
            top_k=50,
            do_sample=True
        )

    # Decode the output
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    model_answer = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    return full_text, model_answer

question = "Is a baby crying?"
full_answer, model_answer = ask_video_chatbot(question)
display_markdown(Markdown(f"**{model_answer}**"))
display_markdown(Markdown("----"))
display_markdown(Markdown(full_answer))

Searching..: 100%|██████████| 3/3 [00:00<00:00, 20560.31it/s]


**Yes, a baby is crying in Match 1 (20s to 25s) [0:20].**

----

system
You are a helpful video assistant. Answer the user's question using ONLY the provided video context. Always cite the exact timestamps in your answer. If the context does not contain the answer, say 'I cannot see that in the video.'
user
Video Context:
- **Match 1 (20s to 25s)** with distance **0.4584**:
- A woman is holding a baby in her arms. The woman is smiling and has her hand on the baby's hand. The baby is crying. The woman is wearing a gray shirt. The background is green.
----
- **Match 2 (25s to 30s)** with distance **0.5928**:
- A woman holds a baby. The woman is smiling at the baby. The baby is looking at the camera. The baby has a pacifier in its mouth.
----
- **Match 3 (45s to 50s)** with distance **0.6306**:
- The video is a short clip showing a baby eating ice cream. The baby is being held by an adult and is positioned in a way that suggests it is eating the ice cream. The baby is making a face of surprise and seems to be enjoying the ice cream. The adult is smiling and appears to be happy about the baby eating the ice cream. The baby is wearing a diaper and is surrounded by a blanket. The background is blurred and is not clear.
----

User Question: Is a baby crying?
model
Yes, a baby is crying in Match 1 (20s to 25s) [0:20].

In [28]:
question = "Are there any babies who refuse to let go of the ice cream when the parent tries to pull it away?"
full_answer, model_answer = ask_video_chatbot(question)
display_markdown(Markdown(f"**{model_answer}**"))
display_markdown(Markdown("----"))
display_markdown(Markdown(full_answer))

Searching..: 100%|██████████| 3/3 [00:00<00:00, 18001.30it/s]


**I cannot see that in the video.**

----

system
You are a helpful video assistant. Answer the user's question using ONLY the provided video context. Always cite the exact timestamps in your answer. If the context does not contain the answer, say 'I cannot see that in the video.'
user
Video Context:
- **Match 1 (50s to 53.1s)** with distance **0.3936**:
- A baby is being held by an adult and eating a spoonful of ice cream. The baby is wearing a striped outfit and the adult is wearing blue jeans. The baby is reaching for the ice cream with its right hand. The adult is smiling down at the baby.
----
- **Match 2 (45s to 50s)** with distance **0.4078**:
- The video is a short clip showing a baby eating ice cream. The baby is being held by an adult and is positioned in a way that suggests it is eating the ice cream. The baby is making a face of surprise and seems to be enjoying the ice cream. The adult is smiling and appears to be happy about the baby eating the ice cream. The baby is wearing a diaper and is surrounded by a blanket. The background is blurred and is not clear.
----
- **Match 3 (0s to 5s)** with distance **0.4162**:
- The video shows a baby girl eating an ice cream bar. The baby is sitting in a baby carrier and is wearing a headband with a large bow on it. The baby's eyes are open and she has a neutral expression. The baby is holding the ice cream bar with both hands and is taking a bite. The ice cream bar is white and has a brown wrapper. The background is blurred.
----

User Question: Are there any babies who refuse to let go of the ice cream when the parent tries to pull it away?
model
I cannot see that in the video.